# Customer Data Profiling

This notebook evaluates the structure and data quality of customer records from the ERP, CRM, and E-commerce source systems.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# DISPLAY SETTINGS
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)


# ============================================================
# LOCATE PROJECT FOLDERS
# ============================================================

working_directory = Path.cwd()

project_root = next(
    (
        path
        for path in [working_directory, *working_directory.parents]
        if (path / "data" / "raw").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "The project root could not be located. "
        "The folder data/raw was not found."
    )

raw_data_directory = project_root / "data" / "raw"
output_directory = project_root / "data" / "processed" / "profiling"

output_directory.mkdir(parents=True, exist_ok=True)


# ============================================================
# DEFINE AND VALIDATE SOURCE FILES
# ============================================================

source_files = {
    "ERP": "erp_customers.csv",
    "CRM": "crm_customers.csv",
    "E-commerce": "ecommerce_customers.csv",
}

missing_files = [
    raw_data_directory / filename
    for filename in source_files.values()
    if not (raw_data_directory / filename).is_file()
]

if missing_files:
    missing_file_list = "\n".join(str(path) for path in missing_files)
    raise FileNotFoundError(
        f"The following required files were not found:\n{missing_file_list}"
    )


# ============================================================
# LOAD RAW DATA
# ============================================================

# All columns are initially loaded as text to preserve identifiers,
# leading zeros, tax IDs, phone numbers, and postal codes.
datasets = {
    source_name: pd.read_csv(
        raw_data_directory / filename,
        dtype="string",
        encoding="utf-8",
    )
    for source_name, filename in source_files.items()
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_text_series(series):
    """Return a column using pandas' nullable string type."""
    return series.astype("string")


def get_blank_mask(series):
    """Identify non-null values containing only empty spaces."""
    text_series = get_text_series(series)

    blank_mask = (
        text_series.notna()
        & text_series.str.strip().eq("")
    )

    return blank_mask.fillna(False)


def get_populated_values(series):
    """Return values that are neither null nor blank."""
    text_series = get_text_series(series)

    populated_mask = (
        text_series.notna()
        & text_series.str.strip().ne("")
    )

    return text_series[populated_mask.fillna(False)].str.strip()


# ============================================================
# DATASET-LEVEL SUMMARY
# ============================================================

dataset_summary_rows = []

for source_name, dataframe in datasets.items():
    row_count = len(dataframe)
    column_count = len(dataframe.columns)
    total_cells = int(dataframe.size)

    missing_cells = int(dataframe.isna().sum().sum())

    blank_cells = sum(
        int(get_blank_mask(dataframe[column]).sum())
        for column in dataframe.columns
    )

    populated_cells = total_cells - missing_cells - blank_cells
    duplicate_rows = int(dataframe.duplicated().sum())

    dataset_summary_rows.append(
        {
            "source": source_name,
            "rows": row_count,
            "columns": column_count,
            "total_cells": total_cells,
            "populated_cells": populated_cells,
            "missing_cells": missing_cells,
            "blank_cells": blank_cells,
            "completeness_pct": round(
                (populated_cells / total_cells) * 100,
                2,
            ) if total_cells else 0,
            "exact_duplicate_rows": duplicate_rows,
            "exact_duplicate_pct": round(
                (duplicate_rows / row_count) * 100,
                2,
            ) if row_count else 0,
        }
    )

dataset_summary = pd.DataFrame(dataset_summary_rows)


# ============================================================
# SCHEMA CONSISTENCY
# ============================================================

all_columns = sorted(
    set().union(
        *(set(dataframe.columns) for dataframe in datasets.values())
    )
)

schema_matrix = pd.DataFrame({"column": all_columns})

for source_name, dataframe in datasets.items():
    schema_matrix[source_name] = schema_matrix["column"].isin(
        dataframe.columns
    )

reference_columns = set(next(iter(datasets.values())).columns)

schemas_match = all(
    set(dataframe.columns) == reference_columns
    for dataframe in datasets.values()
)


# ============================================================
# COLUMN-LEVEL PROFILING
# ============================================================

column_profile_rows = []

for source_name, dataframe in datasets.items():
    total_rows = len(dataframe)

    for column_name in dataframe.columns:
        series = get_text_series(dataframe[column_name])

        missing_count = int(series.isna().sum())
        blank_count = int(get_blank_mask(series).sum())

        populated_values = get_populated_values(series)
        populated_count = len(populated_values)

        exact_distinct_count = int(
            populated_values.nunique(dropna=True)
        )

        normalized_values = populated_values.str.casefold()

        normalized_distinct_count = int(
            normalized_values.nunique(dropna=True)
        )

        rows_in_repeated_values = int(
            normalized_values.duplicated(keep=False).sum()
        )

        value_lengths = populated_values.str.len()

        minimum_length = (
            int(value_lengths.min())
            if not value_lengths.empty
            else None
        )

        maximum_length = (
            int(value_lengths.max())
            if not value_lengths.empty
            else None
        )

        sample_values = " | ".join(
            populated_values.drop_duplicates().head(3).tolist()
        )

        column_profile_rows.append(
            {
                "source": source_name,
                "column": column_name,
                "total_rows": total_rows,
                "populated_count": populated_count,
                "populated_pct": round(
                    (populated_count / total_rows) * 100,
                    2,
                ) if total_rows else 0,
                "missing_count": missing_count,
                "missing_pct": round(
                    (missing_count / total_rows) * 100,
                    2,
                ) if total_rows else 0,
                "blank_count": blank_count,
                "blank_pct": round(
                    (blank_count / total_rows) * 100,
                    2,
                ) if total_rows else 0,
                "exact_distinct_count": exact_distinct_count,
                "normalized_distinct_count": normalized_distinct_count,
                "normalized_uniqueness_pct": round(
                    (
                        normalized_distinct_count
                        / populated_count
                    ) * 100,
                    2,
                ) if populated_count else 0,
                "rows_in_repeated_values": rows_in_repeated_values,
                "minimum_length": minimum_length,
                "maximum_length": maximum_length,
                "sample_values": sample_values,
            }
        )

column_profile = pd.DataFrame(column_profile_rows)


# ============================================================
# BASIC FORMAT VALIDATION
# ============================================================

format_rules = {
    "email": {
        "rule": "Basic email structure",
        "pattern": r"^[^@\s]+@[^@\s]+\.[^@\s]+$",
        "case_sensitive": False,
    },
    "tax_id": {
        "rule": "US EIN format: NN-NNNNNNN",
        "pattern": r"^\d{2}-\d{7}$",
        "case_sensitive": True,
    },
    "postal_code": {
        "rule": "US ZIP format: NNNNN or NNNNN-NNNN",
        "pattern": r"^\d{5}(?:-\d{4})?$",
        "case_sensitive": True,
    },
    "zip_code": {
        "rule": "US ZIP format: NNNNN or NNNNN-NNNN",
        "pattern": r"^\d{5}(?:-\d{4})?$",
        "case_sensitive": True,
    },
}

format_profile_columns = [
    "source",
    "column",
    "rule",
    "populated_count",
    "valid_count",
    "invalid_count",
    "valid_pct",
    "invalid_examples",
]

format_profile_rows = []

for source_name, dataframe in datasets.items():
    for column_name, rule_details in format_rules.items():
        if column_name not in dataframe.columns:
            continue

        populated_values = get_populated_values(
            dataframe[column_name]
        )

        valid_mask = populated_values.str.fullmatch(
            rule_details["pattern"],
            case=rule_details["case_sensitive"],
            na=False,
        )

        valid_count = int(valid_mask.sum())
        invalid_count = int((~valid_mask).sum())
        populated_count = len(populated_values)

        invalid_examples = " | ".join(
            populated_values[~valid_mask]
            .drop_duplicates()
            .head(5)
            .tolist()
        )

        format_profile_rows.append(
            {
                "source": source_name,
                "column": column_name,
                "rule": rule_details["rule"],
                "populated_count": populated_count,
                "valid_count": valid_count,
                "invalid_count": invalid_count,
                "valid_pct": round(
                    (valid_count / populated_count) * 100,
                    2,
                ) if populated_count else 0,
                "invalid_examples": invalid_examples,
            }
        )

format_profile = pd.DataFrame(
    format_profile_rows,
    columns=format_profile_columns,
)


# ============================================================
# COMBINED DATA PREVIEW
# ============================================================

preview_frames = []

for source_name, dataframe in datasets.items():
    source_preview = dataframe.head(3).copy()
    source_preview.insert(0, "profile_source", source_name)
    preview_frames.append(source_preview)

combined_preview = pd.concat(
    preview_frames,
    ignore_index=True,
    sort=False,
)


# ============================================================
# SAVE PROFILING REPORTS
# ============================================================

dataset_summary_path = (
    output_directory / "dataset_summary.csv"
)

schema_matrix_path = (
    output_directory / "schema_matrix.csv"
)

column_profile_path = (
    output_directory / "column_profile.csv"
)

format_profile_path = (
    output_directory / "format_validation.csv"
)

dataset_summary.to_csv(
    dataset_summary_path,
    index=False,
    encoding="utf-8",
)

schema_matrix.to_csv(
    schema_matrix_path,
    index=False,
    encoding="utf-8",
)

column_profile.to_csv(
    column_profile_path,
    index=False,
    encoding="utf-8",
)

format_profile.to_csv(
    format_profile_path,
    index=False,
    encoding="utf-8",
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("DATASET PREVIEW")
display(combined_preview)

print("\nDATASET SUMMARY")
display(dataset_summary)

print(
    "\nSCHEMA CONSISTENCY:",
    "PASS - all sources contain the same columns."
    if schemas_match
    else "REVIEW REQUIRED - source schemas are different.",
)
display(schema_matrix)

print("\nCOLUMN PROFILE")
display(column_profile)

print("\nFORMAT VALIDATION")
if format_profile.empty:
    print("No configured format-validation columns were found.")
else:
    display(format_profile)

print("\nPROFILING REPORTS SAVED TO:")

for report_path in [
    dataset_summary_path,
    schema_matrix_path,
    column_profile_path,
    format_profile_path,
]:
    print(f"- {report_path.relative_to(project_root)}")

DATASET PREVIEW


,profile_source,ground_truth_id,source_system,source_customer_id,company_name,tax_id,email,phone,address_line_1,city,state,postal_code,updated_at
0,ERP,GT-000001,ERP,ERP-000001,"Rodriguez, Figueroa and Sanchez",45-4108603,contact@rodriguezfigueroaand1.com,855-314-0409,181 Johnson Course,East William,AK,74064,2025-03-31
1,ERP,GT-000002,ERP,ERP-000002,"Gardner, Robinson and Lawrence",21-6374122,contact@gardnerrobinsonandla2.com,982-544-1674,65423 Garcia Light,West Melanieview,AS,06196,2026-05-17
2,ERP,GT-000003,ERP,ERP-000003,"Martinez, Nielsen and Miller",34-1166941,contact@martineznielsenandmi3.com,834-570-9459,959 Janet Cape Apt. 413,South Joshuastad,GA,49021,2025-07-30
3,CRM,GT-000001,CRM,CRM-000001,"Rodriguez, Figueroa and Sanchez",<NA>,contact@rodriguezfigueroaand1.com,<NA>,181 Johnson Course,East William,AK,74064,2026-05-07
4,CRM,GT-000002,CRM,CRM-000002,"Gardner, Robinson and Lawrence",<NA>,contact@gardnerrobinsonandla2.com,(982) 544-1674,65423 Garcia Light,West Melanieview,AS,06196,2025-10-19
5,CRM,GT-000003,CRM,CRM-000003,"MARTINEZ, NIELSEN AND MILLER",<NA>,contact@martineznielsenandmi3.com,(834) 570-9459,959 Janet Cape Apt. 413,South Joshuastad,GA,49021,2025-04-08
6,E-commerce,GT-000001,ECOMMERCE,ECM-000001,"Rodriguez, Figueroa and Sanchez",<NA>,CONTACT@RODRIGUEZFIGUEROAAND1.COM,<NA>,181 JOHNSON COURSE,East William,AK,74064,2026-03-16
7,E-commerce,GT-000003,ECOMMERCE,ECM-000003,"Martinez, Nielsen and Miller",<NA>,contact@martineznielsenandmi3.com,+1 834 570 9459,959 Janet Cape Apt. 413,South Joshuastad,GA,49021,2026-04-01
8,E-commerce,GT-000005,ECOMMERCE,ECM-000005,Trujillo Group,<NA>,contact@trujillogroup5.com,+1 611 474 1084,39537 Wilkerson Row Suite 242,Lake Anna,KS,98413,2025-09-25



DATASET SUMMARY


,source,rows,columns,total_cells,populated_cells,missing_cells,blank_cells,completeness_pct,exact_duplicate_rows,exact_duplicate_pct
0,ERP,500,12,6000,6000,0,0,100.00,0,0.0
1,CRM,406,12,4872,4466,406,0,91.67,0,0.0
2,E-commerce,331,12,3972,3527,445,0,88.80,0,0.0



SCHEMA CONSISTENCY: PASS - all sources contain the same columns.


,column,ERP,CRM,E-commerce
0,address_line_1,True,True,True
1,city,True,True,True
2,company_name,True,True,True
3,email,True,True,True
4,ground_truth_id,True,True,True
5,phone,True,True,True
6,postal_code,True,True,True
7,source_customer_id,True,True,True
8,source_system,True,True,True
9,state,True,True,True



COLUMN PROFILE


,source,column,total_rows,populated_count,populated_pct,missing_count,missing_pct,blank_count,blank_pct,exact_distinct_count,normalized_distinct_count,normalized_uniqueness_pct,rows_in_repeated_values,minimum_length,maximum_length,sample_values
0,ERP,ground_truth_id,500,500,100.00,0,0.00,0,0.0,500,500,100.00,0,9,9,GT-000001 | GT-000002 | GT-000003
1,ERP,source_system,500,500,100.00,0,0.00,0,0.0,1,1,0.20,500,3,3,ERP
2,ERP,source_customer_id,500,500,100.00,0,0.00,0,0.0,500,500,100.00,0,10,10,ERP-000001 | ERP-000002 | ERP-000003
3,ERP,company_name,500,500,100.00,0,0.00,0,0.0,500,500,100.00,0,7,33,"Rodriguez, Figueroa and Sanchez | Gardner, Rob..."
4,ERP,tax_id,500,500,100.00,0,0.00,0,0.0,500,500,100.00,0,10,10,45-4108603 | 21-6374122 | 34-1166941
5,ERP,email,500,500,100.00,0,0.00,0,0.0,500,500,100.00,0,18,35,contact@rodriguezfigueroaand1.com | contact@ga...
6,ERP,phone,500,500,100.00,0,0.00,0,0.0,500,500,100.00,0,12,12,855-314-0409 | 982-544-1674 | 834-570-9459
7,ERP,address_line_1,500,500,100.00,0,0.00,0,0.0,500,500,100.00,0,12,32,181 Johnson Course | 65423 Garcia Light | 959 ...
8,ERP,city,500,500,100.00,0,0.00,0,0.0,493,493,98.60,14,7,22,East William | West Melanieview | South Joshua...
9,ERP,state,500,500,100.00,0,0.00,0,0.0,59,59,11.80,500,2,2,AK | AS | GA



FORMAT VALIDATION


,source,column,rule,populated_count,valid_count,invalid_count,valid_pct,invalid_examples
0,ERP,email,Basic email structure,500,500,0,100.00,
1,ERP,tax_id,US EIN format: NN-NNNNNNN,500,500,0,100.00,
2,ERP,postal_code,US ZIP format: NNNNN or NNNNN-NNNN,500,500,0,100.00,
3,CRM,email,Basic email structure,375,375,0,100.00,
4,CRM,tax_id,US EIN format: NN-NNNNNNN,93,52,41,55.91,574781295 | 146194416 | 346442089 | 267945036 ...
5,CRM,postal_code,US ZIP format: NNNNN or NNNNN-NNNN,386,386,0,100.00,
6,E-commerce,email,Basic email structure,316,316,0,100.00,
7,E-commerce,tax_id,US EIN format: NN-NNNNNNN,18,6,12,33.33,927190929 | 247857296 | 920358318 | 252994290 ...
8,E-commerce,postal_code,US ZIP format: NNNNN or NNNNN-NNNN,303,303,0,100.00,



PROFILING REPORTS SAVED TO:
- data\processed\profiling\dataset_summary.csv
- data\processed\profiling\schema_matrix.csv
- data\processed\profiling\column_profile.csv
- data\processed\profiling\format_validation.csv
